In [ ]:
!pip -q install -U optuna optuna-integration chronos-forecasting wandb pyarrow
!pip -q uninstall -y torchao || true          # peft needs torchao>=0.16 OR none
import os, gc, numpy as np, pandas as pd, torch, optuna, wandb
print("optuna", optuna.__version__, "| CUDA:", torch.cuda.is_available())
wandb.login()
os.environ.setdefault("WANDB_PROJECT", "chronos-taxi-optuna")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_PATH   = "/content/drive/MyDrive/Timesfm_Chronos_fine-tune/data/taxi_series.parquet"
BEST_DIR    = "/content/drive/MyDrive/Timesfm_Chronos_fine-tune/chronos_optuna_best"
RESOLUTIONS = ["1h"]                 # start small; [] = all 9 (slower)
os.makedirs(BEST_DIR, exist_ok=True)

df = pd.read_parquet(DATA_PATH)
if RESOLUTIONS:
    df = df[df.resolution.isin(RESOLUTIONS)].copy()
df["ts"] = pd.to_datetime(df["ts"]); df = df.sort_values(["series_id","split","ts"])
def to_dict(sp): return {s: g.sort_values("ts")["value"].to_numpy(np.float32)
                         for s, g in df[df.split==sp].groupby("series_id")}
TRAIN, VAL = to_dict("train"), to_dict("val")
META = df[["series_id","metric","resolution","vendor"]].drop_duplicates().set_index("series_id")
FULL = {s: np.concatenate([TRAIN[s], VAL.get(s, np.array([],np.float32))]) for s in TRAIN}
VAL_START = {s: len(TRAIN[s]) for s in TRAIN}
train_inputs = [{"target": TRAIN[s]} for s in TRAIN if len(TRAIN[s]) >= 512 + 24]
print("series:", len(TRAIN), "| train_inputs:", len(train_inputs))


In [ ]:
CONTEXT, HORIZON, EVAL_MAX_WINDOWS = 512, 24, 150
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def wape(y,p):
    y,p=np.asarray(y,float),np.asarray(p,float); d=np.abs(y).sum()
    return float(np.abs(y-p).sum()/d*100) if d else float("nan")

@torch.no_grad()
def _median_forecast(pipe, ctx_batch, H):
    x = torch.tensor(np.asarray(ctx_batch, np.float32)[:, None, :], dtype=torch.float32)  # (B,1,C) CPU
    q, _ = pipe.predict_quantiles(x, prediction_length=H, quantile_levels=[0.5])
    if isinstance(q, (list, tuple)):
        q = np.stack([a.float().cpu().numpy() if hasattr(a,"cpu") else np.asarray(a,float) for a in q], 0)
    else:
        q = q.float().cpu().numpy() if hasattr(q,"cpu") else np.asarray(q,float)
    B = x.shape[0]
    return np.asarray(q, float).reshape(B, -1)[:, :H]

@torch.no_grad()
def eval_wape(pipe, batch=64):
    windows = []
    for sid, arr in FULL.items():
        vs = VAL_START[sid]
        origins = [o for o in range(vs, len(arr)-HORIZON+1, HORIZON) if o-CONTEXT >= 0]
        if len(origins) > EVAL_MAX_WINDOWS:
            origins = [origins[i] for i in np.linspace(0, len(origins)-1, EVAL_MAX_WINDOWS).astype(int)]
        for o in origins:
            windows.append((sid, arr[o-CONTEXT:o], arr[o:o+HORIZON]))
    per = {}
    for i in range(0, len(windows), batch):
        ch = windows[i:i+batch]
        mp = _median_forecast(pipe, np.stack([c for _,c,_ in ch]), HORIZON)
        for j,(sid,_,tgt) in enumerate(ch):
            per.setdefault(sid, ([],[])); per[sid][0].append(tgt); per[sid][1].append(mp[j])
    vals = [wape(np.concatenate(ys), np.concatenate(ps)) for ys, ps in per.values()]
    return float(np.nanmedian(vals)) if vals else float("nan")   # nan-safe


In [ ]:
from chronos import BaseChronosPipeline
BASE = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-2", device_map=DEVICE,
    torch_dtype=torch.bfloat16 if DEVICE=="cuda" else torch.float32)

LORA_TARGETS = ["self_attention.q","self_attention.v","self_attention.k",
                "self_attention.o","output_patch_embedding.output_layer"]
NUM_STEPS, BATCH = 800, 32          # per-trial budget (raise for deeper search)

def objective(trial):
    lr    = trial.suggest_float("lr", 1e-5, 5e-4, log=True)
    r     = trial.suggest_categorical("lora_r", [4, 8, 16, 32])
    alpha = trial.suggest_categorical("lora_alpha", [8, 16, 32, 64])
    run = wandb.init(project=os.environ["WANDB_PROJECT"], name=f"trial-{trial.number}",
                     config=dict(lr=lr, lora_r=r, lora_alpha=alpha, num_steps=NUM_STEPS), reinit=True)
    try:
        ft = BASE.fit(
            train_inputs, prediction_length=HORIZON, finetune_mode="lora",
            lora_config={"r": r, "lora_alpha": alpha, "target_modules": LORA_TARGETS},
            context_length=CONTEXT, learning_rate=lr, num_steps=NUM_STEPS,
            batch_size=BATCH, disable_tqdm=True)
        w = eval_wape(ft)
        del ft
    except Exception as e:
        wandb.log({"error": str(e)[:200]}); wandb.finish()
        gc.collect(); torch.cuda.empty_cache()
        return 1e6
    if not np.isfinite(w):
        w = 1e6
    wandb.log({"val_wape": w}); wandb.finish()
    gc.collect(); torch.cuda.empty_cache()
    return w


In [ ]:
study = optuna.create_study(direction="minimize",
                            sampler=optuna.samplers.TPESampler(seed=42),
                            study_name="chronos-taxi")
study.optimize(objective, n_trials=12, gc_after_trial=True)
print("BEST WAPE:", round(study.best_value, 3))
print("BEST PARAMS:", study.best_params)


In [ ]:
try:
    import optuna.visualization as vis
    vis.plot_optimization_history(study).show()
    vis.plot_param_importances(study).show()
    vis.plot_parallel_coordinate(study).show()
except Exception as e:
    print("viz:", e)

bp = study.best_params
best = BASE.fit(
    train_inputs, prediction_length=HORIZON, finetune_mode="lora",
    lora_config={"r": bp["lora_r"], "lora_alpha": bp["lora_alpha"], "target_modules": LORA_TARGETS},
    context_length=CONTEXT, learning_rate=bp["lr"], num_steps=NUM_STEPS*2,   # a bit longer for final
    batch_size=BATCH, output_dir=BEST_DIR, finetuned_ckpt_name="checkpoint", disable_tqdm=True)
print("final holdout WAPE:", round(eval_wape(best), 3))
print("saved best checkpoint →", os.path.join(BEST_DIR, "checkpoint"))
